# PySpark: primeros pasos en Databricks
**Formación práctica · 60 minutos · Nivel inicial**

Aprenderás a explorar un DataFrame, filtrar y transformar filas, calcular métricas y combinar datos.

**Requisitos:** conocimientos básicos de Python. No se necesita experiencia con Spark ni archivos externos.

## Guion de la sesión
| Minutos | Actividad |
|---|---|
| 0–5 | Spark, DataFrames y ejecución diferida |
| 5–10 | Crear y explorar datos |
| 10–20 | Seleccionar, filtrar y transformar |
| 20–30 | Agregar y combinar |
| 30–55 | Cuatro ejercicios |
| 55–60 | Repaso y dudas |

## Antes de empezar
Abre este notebook en Google Colab, ejecuta la celda de instalación e inicialización de Spark y continúa de arriba abajo.

Ejecuta las celdas de arriba abajo con **Shift + Enter**. Detente al llegar a los ejercicios: las soluciones están al final. Las celdas de ejercicios contienen comentarios para que «Ejecutar todo» no produzca errores por código incompleto.

Los datos son ficticios y se crean en memoria. No se escriben tablas ni archivos.

In [ ]:
%pip install -q pyspark

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")
         .appName("PySpark_Basicos_60min")
         .getOrCreate())
print("Versión de Spark:", spark.version)


# Parte 1 · Teoría con ejemplos (30 min)
## 1. Spark y DataFrames · 5 min
**Apache Spark** es un motor de procesamiento que puede distribuir el trabajo entre varias máquinas. **PySpark** permite utilizarlo desde Python.

Un **DataFrame** representa datos organizados en filas y columnas, con un esquema que define nombres y tipos. Aunque los datos del taller son pequeños, las operaciones que aprenderemos se utilizan también con grandes volúmenes.

En Google Colab, creamos la sesión **`spark`** al comienzo del notebook.

- Una **transformación**, como `select()` o `filter()`, devuelve un nuevo DataFrame y construye un plan de cálculo.
- Una **acción**, como `show()` o `count()`, solicita un resultado y desencadena la ejecución necesaria.
- Spark utiliza **evaluación diferida**: las transformaciones no calculan inmediatamente todos sus resultados.

Los DataFrames son inmutables: transformar uno no modifica el original. Reasignar una variable solo cambia a qué DataFrame apunta.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# En Colab se reutiliza la sesión creada en la primera celda.
print("Versión de Spark:", spark.version)


## 2. Crear y explorar · 5 min
Trabajaremos con dos DataFrames:

- **`ventas`**: una fila por línea de venta. `id_venta` es único en este ejemplo.
- **`clientes`**: una fila por cliente, con identificador único.

Cada línea tiene producto, categoría, unidades y precio unitario en euros. El cliente 99 no aparece en el catálogo: nos servirá para observar un `left join`.

Usamos un **esquema explícito** para controlar los tipos. `nullable=False` indica que el campo no admite nulos en estos datos de entrada. En este taller usamos `DoubleType` por sencillez; para importes monetarios de producción conviene definir precisión y escala con `DecimalType`.

In [ ]:
esquema_ventas = StructType([
    StructField("id_venta", IntegerType(), False),
    StructField("id_cliente", IntegerType(), False),
    StructField("producto", StringType(), False),
    StructField("categoria", StringType(), False),
    StructField("unidades", IntegerType(), False),
    StructField("precio_unitario", DoubleType(), False),
])

datos_ventas = [
    (1, 1, "Portátil", "Tecnología", 1, 800.0),
    (2, 2, "Ratón", "Tecnología", 2, 25.0),
    (3, 1, "Cuaderno", "Papelería", 5, 4.0),
    (4, 3, "Monitor", "Tecnología", 1, 200.0),
    (5, 2, "Bolígrafo", "Papelería", 10, 1.5),
    (6, 4, "Teclado", "Tecnología", 1, 60.0),
    (7, 3, "Cuaderno", "Papelería", 3, 4.0),
    (8, 1, "Ratón", "Tecnología", 1, 25.0),
    (9, 4, "Mochila", "Accesorios", 2, 30.0),
    (10, 99, "Monitor", "Tecnología", 1, 200.0),
    (11, 2, "Mochila", "Accesorios", 1, 30.0),
    (12, 3, "Teclado", "Tecnología", 2, 60.0),
]
ventas = spark.createDataFrame(datos_ventas, esquema_ventas)

clientes = spark.createDataFrame(
    [(1, "Ana", "Madrid"), (2, "Luis", "Sevilla"),
     (3, "Marta", "Valencia"), (4, "Pablo", "Bilbao"),
     (5, "Elena", "Zaragoza")],
    "id_cliente INT, nombre STRING, ciudad STRING"
)


In [ ]:
ventas.printSchema()
ventas.show(5, truncate=False)
print("Número de líneas de venta:", ventas.count())
print("Columnas:", ventas.columns)


In [ ]:
# show() muestra los datos en Google Colab.
ventas.orderBy("id_venta").show(truncate=False)


**Qué observar:** hay 12 filas y 6 columnas. `show(5)` solo muestra cinco filas; no limita el DataFrame original. Sin `orderBy()`, el orden de las filas no está garantizado.

`show()` solicita datos para mostrarlos. Evita traer todo un conjunto grande a Python con `collect()` o `toPandas()`: el resultado debe caber en la memoria del proceso que lo recibe.

## 3. Seleccionar, filtrar y transformar · 10 min
### Seleccionar columnas
`select()` elige columnas o expresiones. `F.col()` representa una columna de Spark; no es una lista de Python. `alias()` cambia el nombre de una columna en el resultado.

Usamos el alias **`F`** para identificar las funciones de PySpark y evitar confundirlas con funciones de Python.

In [ ]:
seleccion = ventas.select(
    "id_venta", "producto", F.col("precio_unitario").alias("precio_eur")
)
seleccion.orderBy("id_venta").show(4, truncate=False)


### Filtrar y ordenar
`filter()` conserva las filas cuya condición es verdadera. Combina condiciones con **`&`** (y), **`|`** (o) y **`~`** (no), poniendo cada condición entre paréntesis. No uses `and` u `or` de Python con columnas de Spark.

`orderBy()` ordena el resultado; `desc()` indica orden descendente.

In [ ]:
ventas_filtradas = ventas.filter(
    (F.col("categoria") == "Tecnología") & (F.col("unidades") >= 2)
)
ventas_filtradas.orderBy(F.col("precio_unitario").desc(), "id_venta").show()
# Resultado: ventas 12 y 2, en ese orden.


### Crear columnas y aplicar condiciones
`withColumn()` añade una columna o reemplaza otra con el mismo nombre. `when().otherwise()` expresa una condición evaluada para cada fila.

Calcularemos el importe de cada línea y la clasificaremos como «Alta» cuando alcance 100 €. `F.lit()` crea un valor constante dentro de una expresión de Spark.

In [ ]:
ventas_con_importe = ventas.withColumn(
    "importe", F.round(F.col("unidades") * F.col("precio_unitario"), 2)
)
ventas_clasificadas = ventas_con_importe.withColumn(
    "tipo_venta",
    F.when(F.col("importe") >= 100, F.lit("Alta")).otherwise(F.lit("Estándar"))
)
ventas_clasificadas.select("id_venta", "producto", "importe", "tipo_venta").orderBy("id_venta").show()
# ventas sigue teniendo sus 6 columnas originales.
print("Columnas originales:", ventas.columns)


**Pregunta rápida:** ¿`ventas.filter(...)` calcula inmediatamente todas las filas filtradas? No: construye el plan. Al llamar a `show()` se solicita la ejecución necesaria para mostrar resultados.

## 4. Resumir y combinar · 10 min
### Agrupar y agregar
`groupBy()` define los grupos y `agg()` calcula métricas en cada grupo.

- `F.sum("importe")`: suma de importes.
- `F.count("*")`: número de filas, aquí líneas de venta.
- `F.avg("importe")`: importe medio por línea.

Después de agrupar, el resultado contiene una fila por grupo. El importe medio por línea no equivale al precio unitario medio.

In [ ]:
resumen_categoria = (
    ventas_con_importe
    .groupBy("categoria")
    .agg(
        F.count("*").alias("lineas"),
        F.round(F.sum("importe"), 2).alias("importe_total"),
        F.round(F.avg("importe"), 2).alias("importe_medio_linea")
    )
    .orderBy(F.col("importe_total").desc())
)
resumen_categoria.show()
# Tecnología: 1455.0 €; Accesorios: 90.0 €; Papelería: 47.0 €.


### Combinar con un left join
Un **join** combina filas mediante una clave común.

| Tipo | Filas que conserva |
|---|---|
| `inner` | Solo las que tienen coincidencia en ambos lados |
| `left` | Todas las del lado izquierdo y sus coincidencias del derecho |

Añadimos los datos del cliente a cada línea de venta. Al pasar `on="id_cliente"`, la clave común aparece una sola vez en el resultado.

Si la clave del lado derecho tiene duplicados, un join puede multiplicar filas. Aquí `clientes.id_cliente` es único.

In [ ]:
ventas_detalle = ventas_con_importe.join(clientes, on="id_cliente", how="left")
ventas_detalle.select(
    "id_venta", "id_cliente", "nombre", "ciudad", "importe"
).orderBy("id_venta").show()

# La venta del cliente 99 se conserva, con nombre y ciudad NULL.
ventas_detalle.filter(F.col("nombre").isNull()).show()


**Qué observar:** el resultado conserva las 12 líneas. El cliente 99 no tiene nombre ni ciudad. Elena (cliente 5) no aparece porque no tiene ventas y `ventas_con_importe` es el lado izquierdo.

Para buscar nulos utiliza `isNull()` o `isNotNull()`, en lugar de comparar con `None`.

**Pausa antes de practicar:** identifica una transformación, una acción y la clave utilizada en el join.

# Parte 2 · Ejercicios (25 min)
Trabaja a partir de **`ventas`** y **`clientes`**, creados al principio. No necesitas reutilizar los resultados de los ejemplos.

Cada ejercicio indica el nombre de su resultado para facilitar la puesta en común. Puedes usar las pistas si te bloqueas. Las soluciones están en la siguiente sección.

## Ejercicio 1 · Seleccionar, filtrar y ordenar (5 min)
Obtén las ventas de la categoría **Tecnología** con precio unitario **mayor o igual a 60 €**.

1. Conserva `id_venta`, `producto`, `unidades` y `precio_unitario`.
2. Ordena por `precio_unitario` descendente y, en caso de empate, por `id_venta` ascendente.
3. Guarda el resultado en `ej1` y muéstralo.

**Comprobación:** deben aparecer las ventas **1, 4, 10, 6 y 12**, en ese orden.

**Pista:** combina dos condiciones entre paréntesis con `&`.

In [ ]:
# Escribe aquí tu solución.
# ej1 = ...
# ej1.show()


## Ejercicio 2 · Calcular y clasificar (6 min)
Partiendo de todas las filas de `ventas`:

1. Crea `importe` como unidades × precio unitario, redondeado a dos decimales.
2. Crea `tipo_venta`: «Alta» si `importe >= 100`; «Estándar» en el resto.
3. Guarda el resultado en `ej2` y muestra las columnas necesarias para revisarlo.

**Comprobación:** la venta 2 tiene importe **50.0** y tipo «Estándar»; la venta 12 tiene importe **120.0** y tipo «Alta». Hay **4** líneas clasificadas como «Alta».

**Pista:** encadena dos llamadas a `withColumn()` y usa `F.when()`.

In [ ]:
# Escribe aquí tu solución.
# ej2 = ...
# ej2.orderBy("id_venta").show()


## Ejercicio 3 · Calcular ventas por cliente (7 min)
A partir de `ej2`:

1. Agrupa por `id_cliente`.
2. Calcula `numero_lineas` y `total_ventas` (suma de `importe`, redondeada a dos decimales).
3. Ordena de mayor a menor `total_ventas` y guarda el resultado en `ej3`.

**Comprobación:** debes obtener 5 filas. El cliente 1 tiene **3 líneas** y **845.0 €**. La suma de los totales de todos los clientes es **1592.0 €**.

**Pista:** usa `groupBy().agg()`. Aquí contamos líneas, no unidades.

In [ ]:
# Escribe aquí tu solución.
# ej3 = ...
# ej3.show()


## Ejercicio 4 · Añadir nombres sin perder ventas (7 min)
Combina `ej3` con `clientes` mediante un **left join**, manteniendo `ej3` a la izquierda.

1. Conserva `id_cliente`, `nombre`, `numero_lineas` y `total_ventas`.
2. Ordena por `total_ventas` descendente y guarda el resultado en `ej4`.
3. Muestra también qué filas no tienen nombre de cliente.

**Comprobación:** quedan 5 filas. El cliente 99 sigue presente, con nombre nulo y **200.0 €**. Elena no aparece.

**Pista:** `join(..., on="id_cliente", how="left")` e `isNull()`.

In [ ]:
# Escribe aquí tu solución.
# ej4 = ...
# ej4.show()
# ej4.filter(...).show()


# Soluciones · Consultar después de los ejercicios
Estas celdas son ejecutables y reconstruyen los resultados en orden. Ejecuta primero la creación de datos de la parte 1. Se usan variables `sol1` a `sol4` para conservar tus respuestas.

## Solución 1
Primero filtramos para mantener disponible la columna `categoria`; después seleccionamos las columnas solicitadas.

In [ ]:
sol1 = (
    ventas
    .filter((F.col("categoria") == "Tecnología") & (F.col("precio_unitario") >= 60))
    .select("id_venta", "producto", "unidades", "precio_unitario")
    .orderBy(F.col("precio_unitario").desc(), F.col("id_venta").asc())
)
sol1.show()


## Solución 2
La segunda transformación puede utilizar `importe` porque la primera ya lo incorpora al nuevo DataFrame.

In [ ]:
sol2 = (
    ventas
    .withColumn("importe", F.round(F.col("unidades") * F.col("precio_unitario"), 2))
    .withColumn("tipo_venta", F.when(F.col("importe") >= 100, "Alta").otherwise("Estándar"))
)
sol2.select("id_venta", "importe", "tipo_venta").orderBy("id_venta").show()
print("Líneas de tipo Alta:", sol2.filter(F.col("tipo_venta") == "Alta").count())


## Solución 3
`F.count("*")` cuenta filas. Para contar unidades vendidas utilizaríamos `F.sum("unidades")`.

In [ ]:
sol3 = (
    sol2.groupBy("id_cliente")
    .agg(
        F.count("*").alias("numero_lineas"),
        F.round(F.sum("importe"), 2).alias("total_ventas")
    )
    .orderBy(F.col("total_ventas").desc())
)
sol3.show()
# Orden esperado: cliente 1 (845), 3 (332), 99 (200), 4 (120), 2 (95).
sol3.agg(F.round(F.sum("total_ventas"), 2).alias("total_general")).show()


## Solución 4
El `left join` conserva cada cliente con ventas aunque falten sus datos en el catálogo. Volvemos a ordenar después del join: el orden anterior no está garantizado tras combinar.

In [ ]:
sol4 = (
    sol3.join(clientes, on="id_cliente", how="left")
    .select("id_cliente", "nombre", "numero_lineas", "total_ventas")
    .orderBy(F.col("total_ventas").desc())
)
sol4.show()
sol4.filter(F.col("nombre").isNull()).show()


# Cierre · 5 min
## Lo esencial
| Necesidad | Operación |
|---|---|
| Ver estructura y datos | `printSchema()`, `show()` |
| Elegir columnas | `select()` |
| Conservar filas por una condición | `filter()` |
| Añadir o reemplazar una columna | `withColumn()` |
| Aplicar una condición por fila | `F.when().otherwise()` |
| Ordenar filas | `orderBy()` |
| Resumir grupos | `groupBy().agg()` |
| Combinar datos | `join()` |

## Preguntas de repaso
1. ¿Qué diferencia hay entre una transformación y una acción?
2. ¿Por qué usamos `&` en lugar de `and` al combinar condiciones sobre columnas?
3. ¿Qué sucedería con el cliente 99 si el último join fuera `inner`?

**Respuestas:** una transformación construye un nuevo plan y una acción solicita resultados; `&` combina expresiones de columna de Spark; con `inner` desaparecería el cliente 99 y quedarían 4 filas.
